In [2]:
import pandas as pd
# read csv
airports = pd.read_csv("~/Desktop/Sum26_DEPP/airports.csv")
flights = pd.read_csv("~/Desktop/Sum26_DEPP/flights_sample_3m.csv")
# dimension check
print("airports.csv dimension:", airports.shape)
print("flights_sample_3m.csv dimension:", flights.shape)
# inspecting columns and nulls
print("\nairportscolumns:")
print(airports.info())
print("\nflights columns:")
print(flights.info())
print("\nAirports null counts:")
print(airports.isnull().sum())
print("\nFlights null counts:")
print(flights.isnull().sum())

airports.csv dimension: (322, 7)
flights_sample_3m.csv dimension: (3000000, 32)

airportscolumns:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 322 entries, 0 to 321
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   IATA_CODE  322 non-null    object 
 1   AIRPORT    322 non-null    object 
 2   CITY       322 non-null    object 
 3   STATE      322 non-null    object 
 4   COUNTRY    322 non-null    object 
 5   LATITUDE   319 non-null    float64
 6   LONGITUDE  319 non-null    float64
dtypes: float64(2), object(5)
memory usage: 17.7+ KB
None

flights columns:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000000 entries, 0 to 2999999
Data columns (total 32 columns):
 #   Column                   Dtype  
---  ------                   -----  
 0   FL_DATE                  object 
 1   AIRLINE                  object 
 2   AIRLINE_DOT              object 
 3   AIRLINE_CODE             object 
 4   DOT_CODE        

In [13]:
# dealing with missing latitude/longitude in airport dataset first
missing_coords = airports[airports['LATITUDE'].isnull() | airports['LONGITUDE'].isnull()]
print("Missing coordinate rows:")
print(missing_coords[['IATA_CODE', 'AIRPORT', 'CITY', 'STATE']])
# tried to see any duplicate IATA code rows in entire df, but no duplicates, so change to manually impute latitude and longitude
airports.loc[airports['IATA_CODE'] == 'ECP', ['LATITUDE', 'LONGITUDE']] = [30.3582, -85.7956]
airports.loc[airports['IATA_CODE'] == 'PBG', ['LATITUDE', 'LONGITUDE']] = [44.6509, -73.4681]
airports.loc[airports['IATA_CODE'] == 'UST', ['LATITUDE', 'LONGITUDE']] = [29.9593, -81.3397]
# check nulls
print(airports.isnull().sum())

Missing coordinate rows:
Empty DataFrame
Columns: [IATA_CODE, AIRPORT, CITY, STATE]
Index: []
IATA_CODE    0
AIRPORT      0
CITY         0
STATE        0
COUNTRY      0
LATITUDE     0
LONGITUDE    0
dtype: int64


In [14]:
# cancelled no delayed time
flights_clean = flights[flights['CANCELLED'] == 0].copy()
# diverted don't land in original destination, so arrival delay not accurate
flights_clean = flights_clean[flights_clean['DIVERTED'] == 0]
flights_clean.dropna(subset=['CRS_ELAPSED_TIME'], inplace=True)
# have to keep these so fill with 0 later, these rows may indicates non-delayed or delayed with other reasons, can go back and check after join
delay_cols = ['DELAY_DUE_CARRIER', 'DELAY_DUE_WEATHER', 'DELAY_DUE_NAS', 'DELAY_DUE_SECURITY', 'DELAY_DUE_LATE_AIRCRAFT']
flights_clean[delay_cols] = flights_clean[delay_cols].fillna(0)
# non necessary column
flights_clean.drop(columns=['CANCELLATION_CODE'], inplace=True)
# drop the final 2 rows with missing arrival data
flights_clean.dropna(subset=['ARR_TIME'], inplace=True)
# check dim and null
print(flights_clean.shape)
print(flights_clean.isnull().sum())

(2913802, 31)
FL_DATE                    0
AIRLINE                    0
AIRLINE_DOT                0
AIRLINE_CODE               0
DOT_CODE                   0
FL_NUMBER                  0
ORIGIN                     0
ORIGIN_CITY                0
DEST                       0
DEST_CITY                  0
CRS_DEP_TIME               0
DEP_TIME                   0
DEP_DELAY                  0
TAXI_OUT                   0
WHEELS_OFF                 0
WHEELS_ON                  0
TAXI_IN                    0
CRS_ARR_TIME               0
ARR_TIME                   0
ARR_DELAY                  0
CANCELLED                  0
DIVERTED                   0
CRS_ELAPSED_TIME           0
ELAPSED_TIME               0
AIR_TIME                   0
DISTANCE                   0
DELAY_DUE_CARRIER          0
DELAY_DUE_WEATHER          0
DELAY_DUE_NAS              0
DELAY_DUE_SECURITY         0
DELAY_DUE_LATE_AIRCRAFT    0
dtype: int64


In [22]:
# convert to datetime
flights_clean['FL_DATE'] = pd.to_datetime(flights_clean['FL_DATE'])

def format_time(val):
    val_str = str(int(val)).zfill(4)
    return f"{val_str[:2]}:{val_str[2:]}"

flights_clean['SCHED_DEP_TIMESTAMP'] = pd.to_datetime(
    flights_clean['FL_DATE'].dt.strftime('%Y-%m-%d') + ' ' + flights_clean['CRS_DEP_TIME'].apply(format_time)
)
# round to nearest hour
flights_clean['WEATHER_JOIN_HOUR'] = flights_clean['SCHED_DEP_TIMESTAMP'].dt.round('h')
# change to categorical
categorical_cols = ['AIRLINE', 'AIRLINE_CODE', 'ORIGIN', 'DEST', 'ORIGIN_CITY', 'DEST_CITY']
for col in categorical_cols:
    flights_clean[col] = flights_clean[col].astype('category')
# downcast delay minutes to integers
int_cols = [
    'DEP_DELAY', 'ARR_DELAY', 'TAXI_OUT', 'TAXI_IN', 'AIR_TIME', 'DISTANCE', 'CRS_ELAPSED_TIME', 'ELAPSED_TIME',
    'DELAY_DUE_CARRIER', 'DELAY_DUE_WEATHER', 'DELAY_DUE_NAS', 'DELAY_DUE_SECURITY', 'DELAY_DUE_LATE_AIRCRAFT'
]
for col in int_cols:
    flights_clean[col] = flights_clean[col].astype('int32')
# make sure these columns entry all uppercase
flights_clean['ORIGIN'] = flights_clean['ORIGIN'].str.upper()
airports['IATA_CODE'] = airports['IATA_CODE'].str.upper().str.strip()
# standardize text columns in airports
text_cols = ['AIRPORT', 'CITY', 'STATE', 'COUNTRY']
for col in text_cols:
    airports[col] = airports[col].str.title().str.strip()

In [25]:
# check iata duplicates since primary key
duplicates = airports[airports.duplicated(subset=['IATA_CODE'], keep=False)]

if not duplicates.empty:
    print(duplicates)
else:
    print("no duplicates")

no duplicates


In [26]:
# join coordinates for the departure airport
flights_clean = flights_clean.merge(
    airports[['IATA_CODE', 'LATITUDE', 'LONGITUDE']], 
    left_on='ORIGIN', 
    right_on='IATA_CODE', 
    how='left'
).rename(columns={'LATITUDE': 'ORIGIN_LAT', 'LONGITUDE': 'ORIGIN_LON'}).drop(columns=['IATA_CODE'])

# join coordinates for the destination airport
flights_clean = flights_clean.merge(
    airports[['IATA_CODE', 'LATITUDE', 'LONGITUDE']], 
    left_on='DEST', 
    right_on='IATA_CODE', 
    how='left'
).rename(columns={'LATITUDE': 'DEST_LAT', 'LONGITUDE': 'DEST_LON'}).drop(columns=['IATA_CODE'])

In [27]:
flights_clean

,FL_DATE,AIRLINE,AIRLINE_DOT,AIRLINE_CODE,DOT_CODE,FL_NUMBER,ORIGIN,ORIGIN_CITY,DEST,DEST_CITY,...,DELAY_DUE_WEATHER,DELAY_DUE_NAS,DELAY_DUE_SECURITY,DELAY_DUE_LATE_AIRCRAFT,SCHED_DEP_TIMESTAMP,WEATHER_JOIN_HOUR,ORIGIN_LAT,ORIGIN_LON,DEST_LAT,DEST_LON
0,2019-01-09,United Air Lines Inc.,United Air Lines Inc.: UA,UA,19977,1562,FLL,"Fort Lauderdale, FL",EWR,"Newark, NJ",...,0,0,0,0,2019-01-09 11:55:00,2019-01-09 12:00:00,26.07258,-80.15275,40.69250,-74.16866
1,2022-11-19,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790,1149,MSP,"Minneapolis, MN",SEA,"Seattle, WA",...,0,0,0,0,2022-11-19 21:20:00,2022-11-19 21:00:00,44.88055,-93.21692,47.44898,-122.30931
2,2022-07-22,United Air Lines Inc.,United Air Lines Inc.: UA,UA,19977,459,DEN,"Denver, CO",MSP,"Minneapolis, MN",...,0,0,0,0,2022-07-22 09:54:00,2022-07-22 10:00:00,39.85841,-104.66700,44.88055,-93.21692
3,2023-03-06,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790,2295,MSP,"Minneapolis, MN",SFO,"San Francisco, CA",...,0,24,0,0,2023-03-06 16:09:00,2023-03-06 16:00:00,44.88055,-93.21692,37.61900,-122.37484
4,2020-02-23,Spirit Air Lines,Spirit Air Lines: NK,NK,20416,407,MCO,"Orlando, FL",DFW,"Dallas/Fort Worth, TX",...,0,0,0,0,2020-02-23 18:40:00,2020-02-23 19:00:00,28.42889,-81.31603,32.89595,-97.03720
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2913797,2022-11-13,American Airlines Inc.,American Airlines Inc.: AA,AA,19805,1522,JAX,"Jacksonville, FL",CLT,"Charlotte, NC",...,0,0,0,0,2022-11-13 17:42:00,2022-11-13 18:00:00,30.49406,-81.68786,35.21401,-80.94313
2913798,2022-11-02,American Airlines Inc.,American Airlines Inc.: AA,AA,19805,1535,ORD,"Chicago, IL",AUS,"Austin, TX",...,0,0,0,0,2022-11-02 13:00:00,2022-11-02 13:00:00,41.97960,-87.90446,30.19453,-97.66987
2913799,2022-09-11,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790,2745,HSV,"Huntsville, AL",ATL,"Atlanta, GA",...,36,0,0,0,2022-09-11 05:34:00,2022-09-11 06:00:00,34.64045,-86.77311,33.64044,-84.42694
2913800,2019-11-13,Republic Airline,Republic Airline: YX,YX,20452,6134,BOS,"Boston, MA",LGA,"New York, NY",...,0,0,0,0,2019-11-13 16:00:00,2019-11-13 16:00:00,42.36435,-71.00518,40.77724,-73.87261


In [28]:
flights_clean.to_csv("~/Desktop/Sum26_DEPP/flights_sample_3m_clean.csv", index=False)